# Analyse a GloFAS flood forecast signal


[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/Simow-az/2026-icar-glofas-training/main?labpath=content/tutorials/analyse-glofas-forecast-signal.ipynb)
[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Simow-az/2026-icar-glofas-training/blob/main/content/tutorials/analyse-glofas-forecast-signal.ipynb)
[![github](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Simow-az/2026-icar-glofas-training/blob/main/content/tutorials/analyse-glofas-forecast-signal.ipynb)

:::{note}
This notebook introduces a forecast-oriented workflow using a synthetic ensemble so that you can practise the analysis steps without waiting for a download.
:::

This tutorial shows how to prepare a public GloFAS forecast request and then turn a small ensemble-style dataset into practical forecast indicators such as the ensemble median and exceedance probability.


## Learning objectives 🎯


By the end of this notebook you will be able to:

- prepare a CDS request for GloFAS forecast discharge data;
- work with forecast lead times and ensemble members in `xarray`;
- derive simple flood-awareness indicators from an ensemble time series.


## Prepare your environment


The optional CDS step uses `cdsapi`, while the analysis uses the Python packages already declared in `environment.yml`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

try:
    import cdsapi
except ImportError:
    cdsapi = None


## Define a forecast request


In [ ]:
dataset = "cems-glofas-forecast"
output_path = Path("data/glofas-forecast-example.grib")
request = {
    "system_version": ["operational"],
    "hydrological_model": ["lisflood"],
    "product_type": ["control_forecast", "ensemble_perturbed_forecasts"],
    "variable": ["river_discharge_in_the_last_24_hours"],
    "year": ["2024"],
    "month": ["03"],
    "day": ["15"],
    "leadtime_hour": [str(hour) for hour in range(24, 241, 24)],
    "data_format": "grib",
    "download_format": "unarchived",
}
request


Forecast requests typically combine the control forecast with the perturbed ensemble members so that you can estimate both the central signal and the spread.


## Optional CDS download


In [ ]:
download_from_cds = False

if download_from_cds:
    if cdsapi is None:
        raise ImportError("Install cdsapi before enabling the download step.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    client = cdsapi.Client()
    client.retrieve(dataset, request, str(output_path))
    print(f"Saved example file to {output_path}")
else:
    print("Skipping the live download. Enable `download_from_cds` when you have CDS access.")


## Build a practice forecast cube


In [ ]:
leadtimes = np.arange(1, 11)
members = ["member_{:02d}".format(member) for member in range(1, 12)]

baseline = np.linspace(180, 420, leadtimes.size)
spread = np.linspace(10, 75, leadtimes.size)
member_offsets = np.linspace(-1.5, 1.5, len(members))[:, None]
ensemble_values = baseline + member_offsets * spread

forecast_ds = xr.Dataset(
    data_vars={
        "river_discharge_in_the_last_24_hours": (("member", "leadtime_day"), ensemble_values),
    },
    coords={
        "member": members,
        "leadtime_day": leadtimes,
    },
)
forecast_ds


## Calculate ensemble indicators


In [ ]:
discharge = forecast_ds["river_discharge_in_the_last_24_hours"]
alert_threshold = 380.0

median_signal = discharge.median(dim="member")
exceedance_probability = (discharge > alert_threshold).mean(dim="member") * 100

xr.Dataset({
    "ensemble_median": median_signal,
    "probability_above_threshold": exceedance_probability,
})


## Visualise the evolving risk


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

discharge.transpose("leadtime_day", "member").plot(ax=axes[0], add_legend=False, alpha=0.4)
median_signal.plot(ax=axes[0], color="black", linewidth=2, label="Ensemble median")
axes[0].axhline(alert_threshold, color="crimson", linestyle="--", label="Alert threshold")
axes[0].set_ylabel("River discharge [m³/s]")
axes[0].set_title("Example forecast hydrograph")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

exceedance_probability.plot(ax=axes[1], color="navy", marker="o")
axes[1].set_ylabel("Probability [%]")
axes[1].set_xlabel("Lead time [days]")
axes[1].set_title("Probability of exceeding the alert threshold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Interpret the signal


In [ ]:
high_risk_days = exceedance_probability.where(exceedance_probability >= 50, drop=True)
high_risk_days.to_series().round(1)


## Take home messages 📌


- A forecast workflow is more informative when you look at both the central estimate and the ensemble spread.
- Simple indicators such as threshold exceedance probability help translate raw discharge forecasts into operational signals.
- You can reuse the same `xarray` patterns on real GloFAS forecast files after replacing the synthetic practice cube.
